# Robot omni

## Wstęp

Aby robot mógł dojechać do zadanego punktu, musi znać własne położenie. Żaden z czujników robota nie mierzy go jednak bezpośrednio i bez błędu. Celem laboratorium jest wyznaczenie położenia robota omnikierunkowego poruszającego się w pudle 1 × 1 m trzema metodami i porównanie ich w czasie rzeczywistym:

* **odometria (odometry)** - całkowanie obrotów kół; estymata jest gładka, ale jej błąd rośnie wraz z przejechaną drogą,
* **dopasowanie skanów LiDAR-u (scan matching)** - położenie wyznaczane względem mapy budowanej przez robota z kolejnych skanów; w niewielkim pomieszczeniu praktycznie nie dryfuje, ale jest obarczone szumem,
* **filtr Kalmana (Kalman filter)** - łączy obie metody, przejmując gładkość odometrii i brak dryfu LiDAR-u.

Laboratorium składa się z dwóch części. Pierwsza dotyczy modelu ruchu: kinematyka napędu zostanie wyprowadzona analitycznie, a następnie dopasowana do pomiarów; na końcu tej części zbadamy dryf odometrii podczas jazdy i obrotów. Druga część obejmuje czujniki i filtr Kalmana, najpierw w jednym wymiarze z czujnikiem ultradźwiękowym, a następnie w postaci filtru położenia działającego w bibliotece robota.

![Robot w pudle 1 × 1 m: LiDAR o polu widzenia 240° (z martwą strefą z tyłu) oraz trzy czujniki ultradźwiękowe skierowane do przodu i na boki](img/robot_in_box.png)

Wykorzystywane biblioteki: [NumPy](https://numpy.org/) i [pandas](https://pandas.pydata.org/) do obliczeń, [Bokeh](https://docs.bokeh.org/) do wykresów aktualizowanych w trakcie jazdy, [pykalman](https://pykalman.github.io/) jako referencyjna implementacja filtru Kalmana oraz `omni_drive`, biblioteka sterująca robotem.

**Bezpieczeństwo.** Przed uruchomieniem komórki, która porusza robotem, należy ustawić go na środku pudła, przodem do ściany. Ruch należy zlecać metodą `await robot.drive_for(...)`, która zatrzymuje robota po zadanym czasie, również w przypadku przerwania komórki (■). Metoda `robot.set_vector(...)` utrzymuje zadany ruch aż do kolejnego polecenia. Robota można w każdej chwili zatrzymać wywołaniem `robot.stop()`.

In [ ]:
import asyncio
import math

import numpy as np
from bokeh.io import output_notebook
from omni_drive import Angle, DriveVector, Localizer, OmniDrive, Pose, PoseNoise, Time
from omni_drive.ui import LidarView, LivePlot, UltrasonicView, plot_series

output_notebook(hide_banner=True)

## Połączenie z robotem

Notebook działa na komputerze robota (Raspberry Pi), z którym łączymy się przez sieć Wi-Fi. Gotowość robota sygnalizuje zielona dioda Raspberry Pi migająca w rytmie bicia serca.

`OmniDrive()` nawiązuje połączenie z dwiema usługami: sterownikiem kół (ESP32), który co 50 ms przesyła odometrię, oraz LiDAR-em, który co 100 ms przesyła skan. `robot.print_status()` pozwala sprawdzić, czy obie usługi działają. Jeśli komórka zgłosi brak telemetrii, należy odczekać kilka sekund i uruchomić ją ponownie. Brak skanów z LiDAR-u należy zgłosić prowadzącemu.

Polecenia ruchu podawane są w układzie robota: kąt `0°` oznacza zawsze jego aktualny przód (`robot.field_centric = False`). Etykiety wykresów i komunikaty biblioteki są w języku angielskim: *Odometry* oznacza odometrię, *Measured* odczyt czujnika, a *Expected* odczyt oczekiwany.

Notebook można uruchomić również bez robota, korzystając z symulatora. W terminalu należy wywołać `OMNI_DRIVE_IPC=tcp://127.0.0.1:5650 python -m omni_drive.simulator`, a w pierwszej komórce notebooka, przed importami, dopisać `import os; os.environ["OMNI_DRIVE_IPC"] = "tcp://127.0.0.1:5650"`.

In [ ]:
robot = OmniDrive()
robot.field_centric = False
await asyncio.sleep(1.0)
robot.print_status()

## Część 1: Model ruchu

### Kinematyka napędu

Robot ma trzy **koła omnikierunkowe (omni wheels)** rozmieszczone co 120° w odległości $L = 0{,}1$ m od środka. Na obwodzie każdego koła znajdują się bierne rolki. Koło napędza więc robota wyłącznie w kierunku stycznym do okręgu, na którym leżą koła, a w kierunku prostopadłym przetacza się swobodnie na rolkach. Dzięki temu robot może poruszać się w dowolnym kierunku i jednocześnie obracać wokół własnej osi; taki napęd nazywamy **holonomicznym (holonomic)**.

Animacja poniżej pokazuje oba rodzaje ruchu koła. Gdy silnik obraca kołem, rolki pozostają nieruchome względem piasty i koło toczy się jak zwykłe koło. Gdy koło jest przesuwane w kierunku osi, piasta stoi, a obracają się wyłącznie rolki.

![Koło omnikierunkowe: ruch w kierunku napędu, a następnie swobodny ruch na rolkach. Model koła kiddo_omniwheel (makerspet, Apache 2.0)](img/omni_wheel_rolling.gif)

![Rozmieszczenie kół robota i kierunki, w których każde z nich napędza platformę](img/wheel_forces_top.png)

Koło ustawione pod kątem $\beta_i$ napędza robota w kierunku $(-\sin\beta_i, \cos\beta_i)$, zatem jego obwód porusza się z prędkością

$$
w_i = -\sin(\beta_i)\, v_x + \cos(\beta_i)\, v_y + L\, \omega,
$$

gdzie $(v_x, v_y)$ jest prędkością robota w jego własnym układzie (oś $x$ do przodu, oś $y$ w lewo), a $\omega$ prędkością obrotu. W naszym robocie $\beta_1 = 60°$, $\beta_2 = 180°$, $\beta_3 = 300°$. Trzy równania zapisane w postaci macierzowej tworzą **kinematykę odwrotną (inverse kinematics)**:

$$
\begin{bmatrix} w_1 \\ w_2 \\ w_3 \end{bmatrix}
=
\underbrace{\begin{bmatrix}
-\sin\beta_1 & \cos\beta_1 & L \\
-\sin\beta_2 & \cos\beta_2 & L \\
-\sin\beta_3 & \cos\beta_3 & L
\end{bmatrix}}_{\mathbf{M}}
\begin{bmatrix} v_x \\ v_y \\ \omega \end{bmatrix}
$$

Biblioteka wykonuje to przeliczenie 50 razy na sekundę, zamieniając polecenie `set_vector` na prędkości kół. Zależność odwrotną opisuje **kinematyka prosta (forward kinematics)**, która na podstawie obrotów kół zmierzonych enkoderami wyznacza ruch robota: $\mathbf{v} = \mathbf{M}^{-1}\mathbf{w}$. Na kinematyce prostej opiera się odometria. Wyprowadzenie dla dowolnej liczby kół zawiera rozdział 13.2 książki [Modern Robotics](https://hades.mech.northwestern.edu/images/7/7f/MR.pdf) (K. Lynch, F. Park).

Odometria całkuje kinematykę prostą. Jeżeli od poprzedniego kroku obwody kół przebyły drogi $\Delta s_1, \Delta s_2, \Delta s_3$, to $[\Delta x, \Delta y, \Delta\theta]^T = \mathbf{M}^{-1} [\Delta s_1, \Delta s_2, \Delta s_3]^T$ jest przesunięciem w układzie robota. Do układu świata przesunięcie obraca się o kąt robota w połowie kroku, $\bar\theta = \theta + \Delta\theta / 2$:

$$
x \leftarrow x + \Delta x \cos\bar\theta - \Delta y \sin\bar\theta, \qquad
y \leftarrow y + \Delta x \sin\bar\theta + \Delta y \cos\bar\theta, \qquad
\theta \leftarrow \theta + \Delta\theta
$$

W ten sam sposób, 100 razy na sekundę, odometrię wyznacza sterownik robota.

**Zadanie 1**

1. Zbuduj macierz `M` (3 × 3) dla kątów zapisanych w `WHEEL_ANGLES`. Funkcje `np.sin` i `np.cos` przyjmują argument w radianach.
2. Uzupełnij funkcję `odometry_step(pose, ds)`. Funkcja przyjmuje **pozę (pose)**, czyli położenie i orientację robota (obiekt `Pose(x, y, yaw)`), oraz przesunięcia obwodów kół `ds` (trzy wartości w metrach) i zwraca nową pozę. Przesunięcie w układzie robota wyznacz funkcją [`np.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html), która rozwiązuje układ równań bez jawnego odwracania macierzy.

Komórka pod zadaniem sprawdza rozwiązanie na przejeździe 1 m na wprost w krokach po 2 mm. Przy idealnych kołach odometria powinna wskazać $(1, 0, 0°)$. Następnie przejazd jest powtarzany dla koła 1 o 2% większego, niż zakłada model.

In [ ]:
L = 0.1
WHEEL_ANGLES = np.radians([60, 180, 300])

M = ...


def odometry_step(pose: Pose, ds: np.ndarray) -> Pose: ...

In [ ]:
def drive_straight(wheel_scale):
    pose = Pose()
    for _ in range(500):
        pose = odometry_step(pose, wheel_scale * (M @ [0.002, 0.0, 0.0]))
    return pose


for name, scale in (("exact wheels", [1.0, 1.0, 1.0]), ("wheel 1 +2%", [1.02, 1.0, 1.0])):
    p = drive_straight(np.array(scale))
    print(f"{name:14s} x = {p.x:.3f} m, y = {p.y * 100:+.1f} cm, yaw = {math.degrees(p.yaw):+.1f}°")

Różnica 2% w promieniu jednego koła powoduje po przejechaniu metra odchylenie o kilka centymetrów w bok i kilka stopni obrotu, i to w modelu pozbawionym poślizgu.

### Model ruchu dopasowany do pomiarów

Macierz $\mathbf{M}$ wyprowadzono przy założeniu idealnych kół: równych promieni, kątów dokładnie co 120° i braku poślizgu. Rzeczywisty robot odbiega od tego modelu, a każda rozbieżność bezpośrednio wpływa na odometrię. Alternatywne podejście nie wymaga założeń o konstrukcji: robot wykonuje serię losowych ruchów, LiDAR mierzy rzeczywiste przemieszczenie, a model dopasowujemy do pomiarów **metodą najmniejszych kwadratów (least squares)**.

Szukamy macierzy $\mathbf{A}$ (3 × 3), dla której iloczyn $\mathbf{A}\, \mathbf{w}$ najlepiej przybliża prędkość robota $\mathbf{v}$ (w m/s i rad/s), gdzie $\mathbf{w}$ to polecenia dla kół w jednostkach znormalizowanych (od −1 do 1). Po ułożeniu $N$ pomiarów w wiersze macierzy $\mathbf{W}$ ($N \times 3$) i $\mathbf{V}$ ($N \times 3$) otrzymujemy problem $\min_{\mathbf{A}} \lVert \mathbf{W}\mathbf{A}^T - \mathbf{V} \rVert^2$, który rozwiązuje funkcja [`np.linalg.lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html).

Model analityczny przewiduje $\mathbf{A} = s_{max}\, \mathbf{M}^{-1}$, gdzie $s_{max} \approx 0{,}39$ m/s jest prędkością obwodu koła przy poleceniu `1.0`. Wartość ta wynika z parametrów enkoderów: maksymalnie 175 impulsów na 100 ms, 990 impulsów na obrót, promień koła 35 mm.

Komórka poniżej uruchamia `Localizer`, który śledzi położenie robota w tle (szczegóły w dalszej części), i wykonuje 10 par ruchów: losowe prędkości kół, a następnie te same prędkości z przeciwnym znakiem, tak aby robot wrócił w okolice środka pudła. Każdy ruch trwa 1,4 s, a prędkość mierzona jest LiDAR-em w ostatniej sekundzie, gdy silniki osiągnęły już zadaną prędkość. Długość okna pomiarowego ma znaczenie, ponieważ LiDAR wyznacza położenie 10 razy na sekundę i każdy koniec okna znany jest z dokładnością około 0,1 s. Przed uruchomieniem komórki ustaw robota na środku pudła.

In [ ]:
localizer = Localizer(robot).start()
await localizer.wait_ready()
rng = np.random.default_rng(0)
W, V = [], []

for _ in range(10):
    w = rng.uniform(-0.2, 0.2, size=3)
    for sign in (1, -1):
        with robot.batch():
            robot.wheel_1.speed, robot.wheel_2.speed, robot.wheel_3.speed = sign * w
        try:
            await asyncio.sleep(0.4)
            start = localizer.estimate.lidar
            await asyncio.sleep(1.0)
            moved = localizer.estimate.lidar.relative_to(start)
        finally:
            robot.stop()
        W.append(sign * w)
        V.append([moved.x, moved.y, moved.yaw])
    await asyncio.sleep(0.3)

W, V = np.array(W), np.array(V)
print(f"samples: {len(W)}")

**Zadanie 2**

1. Dopasuj macierz `A_fit` metodą najmniejszych kwadratów. Pierwszy element wyniku `np.linalg.lstsq(W, V, rcond=None)` to $\mathbf{A}^T$.
2. Wyznacz model analityczny `A_model` $= s_{max}\, \mathbf{M}^{-1}$ (macierz `M` z Zadania 1, funkcja `np.linalg.inv`).
3. Wypisz obie macierze oraz względną różnicę każdego wiersza: `np.linalg.norm(A_fit - A_model, axis=1) / np.linalg.norm(A_model, axis=1)`. Podobna różnica we wszystkich wierszach wskazuje na błędnie oszacowane $s_{max}$, a różnica skupiona w wierszu $\omega$ na niedokładny rozstaw kół $L$.

In [ ]:
S_MAX = 0.39

...

Wynik dopasowania służy do konfiguracji modelu ruchu. Jeżeli $\mathbf{A}_{fit} \approx s \mathbf{M}^{-1}$, to iloczyn $\mathbf{A}_{fit}\mathbf{M}$ jest bliski $s \mathbf{I}$, a średnia z jego przekątnej jest rzeczywistą prędkością obwodu koła przy poleceniu `1.0`. Kolejne komórki wyznaczają czas jazdy na podstawie tej dopasowanej wartości zamiast wartości 0,39 wyliczonej z parametrów enkoderów.

In [ ]:
S_MAX = float(np.mean(np.diag(A_fit @ M)))
print(f"fitted S_MAX: {S_MAX:.3f} m/s")

### Trzy estymaty położenia

Odometria całkuje każdy błąd modelu i nie zawiera mechanizmu korekcji. Na zamkniętej trasie część błędu może się przypadkowo znieść, nie jest to jednak korekcja. Drugim źródłem położenia jest LiDAR [Hokuyo URG-04LX-UG01](https://www.hokuyo-aut.jp/search/single.php?serial=166), który 10 razy na sekundę mierzy około 680 odległości w polu widzenia 240°, w zakresie do 5,6 m, z dokładnością około 3 cm.

![Robot Robotino z LiDAR-em Hokuyo URG-04LX-UG01. Fot. Robodoc9, CC BY-SA 3.0, Wikimedia Commons](img/robotino_hokuyo.jpg)

Z kolejnych skanów biblioteka buduje **mapę zajętości (occupancy grid)**: siatkę komórek 3 × 3 cm, z których każda przechowuje prawdopodobieństwo zajętości. Każdy nowy skan jest **dopasowywany do mapy (scan-to-map matching)**: algorytm wyznacza takie położenie robota, przy którym punkty skanu trafiają w zajęte komórki. Ten sam schemat stosują Hector SLAM i Google Cartographer:

> Kohlbrecher, Stefan, et al. *"A flexible and scalable SLAM system with full 3D motion estimation."* IEEE International Symposium on Safety, Security, and Rescue Robotics. 2011. [link](https://ieeexplore.ieee.org/document/6106777)

> Hess, Wolfgang, et al. *"Real-time loop closure in 2D LIDAR SLAM."* IEEE International Conference on Robotics and Automation (ICRA). 2016. [link](https://research.google.com/pubs/archive/45466.pdf)

Animacja poniżej przedstawia działanie Hector SLAM. Kolorem szarym oznaczono obszar wolny, czarnym ściany, a zieloną linią trajektorię robota; mapa rozrasta się z każdym kolejnym skanem.

![Hector SLAM buduje mapę zajętości ze skanów LiDAR-u. Nagranie: Auledas, CC BY-SA 4.0, Wikimedia Commons (przycięte i przyspieszone 4×)](img/hector_slam.gif)

Położenie z LiDAR-u nie dryfuje w taki sposób jak odometria, ponieważ każdy skan porównywany jest z tą samą mapą. Sama mapa również może kumulować błąd, jednak w pudle 1 × 1 m jest on rzędu milimetrów. Estymata ta jest natomiast zaszumiona, a w przypadku nieudanego dopasowania, np. przy zbyt szybkim obrocie, robot chwilowo traci lokalizację. Trzecią estymatą jest filtr Kalmana łączący obie metody, omówiony w części 2.

`Localizer` wyznacza wszystkie trzy estymaty w tle, we wspólnym układzie odniesienia. Jego początkiem jest położenie robota w chwili uruchomienia (lub wywołania `localizer.reset()`), oś $x$ wskazuje przód robota, a oś $y$ jego lewą stronę. `LidarView` przedstawia estymaty na żywo jako trzy znaczniki położenia robota: pomarańczowy odpowiada odometrii, niebieski LiDAR-owi, a zielony filtrowi Kalmana. Elipsa wokół zielonego znacznika obejmuje dwa odchylenia standardowe niepewności położenia wyznaczonej przez filtr. Kliknięcie pozycji w legendzie ukrywa odpowiadającą jej serię.

In [ ]:
localizer.reset()
view = LidarView(localizer=localizer).show()

Kolejne zadanie pozwala zmierzyć tempo narastania błędu odometrii.

**Zadanie 3**

1. Uzupełnij funkcję `drive_square(side, velocity)`, która realizuje przejazd po kwadracie o boku `side` metrów bez obrotu robota: kolejno do przodu, w lewo, do tyłu i w prawo (`Angle.deg(0)`, `90`, `180`, `270`). Użyj `await robot.drive_for(DriveVector(velocity, heading), Time.s(...))`. Przy `velocity = 1.0` robot porusza się z prędkością około `S_MAX` m/s.
2. Wykonaj 3 okrążenia kwadratu o boku 20 cm z `velocity = 0.3` i obserwuj estymaty położenia na powyższym wykresie.
3. Oblicz, o ile centymetrów i stopni odometria (`localizer.estimate.odometry`) różni się od LiDAR-u (`localizer.estimate.lidar`). Przydatne będą metody `Pose.distance_to` i `Pose.relative_to`.

In [ ]:
async def drive_square(side, velocity): ...


view.reset_pose()
await asyncio.sleep(0.5)
...

Estymaty zapisane podczas jazdy zwraca `localizer.dataframe()` (20 wierszy na sekundę). Poniższy wykres przedstawia zmianę odległości między odometrią a LiDAR-em. Na zamkniętej trasie część błędu odometrii się znosi; podczas jazdy w jednym kierunku błąd byłby większy.

In [ ]:
df = localizer.dataframe()
gap = np.hypot(df.odom_x - df.lidar_x, df.odom_y - df.lidar_y)
plot_series(
    df.t,
    {"odometry - LiDAR [m]": gap},
    title="Odometry drift relative to LiDAR",
    y_label="distance [m]",
)

### Obrót

Dotychczas robot poruszał się bez zmiany orientacji. Trzecia składowa `DriveVector`, `omega`, steruje obrotem niezależnie od ruchu postępowego. Wartość `omega = 1.0` oznacza, że wszystkie koła obracają się z pełną prędkością w tym samym kierunku; robot obraca się wtedy z prędkością $\omega_{max} = s_{max} / L$, czyli około 3,9 rad/s.

![Jazda na wprost, obrót w miejscu i jazda z obrotem. Prędkości kół wynikają z kinematyki odwrotnej](img/kiwi_motion.gif)

Przy jeździe z obrotem stosuje się tryb **field-centric**. W trybie domyślnym kąt `0°` oznacza aktualny przód robota, więc kierunek jazdy obraca się razem z robotem. Po ustawieniu `robot.field_centric = True` biblioteka na bieżąco obraca polecenie o kąt robota, dzięki czemu `0°` oznacza kierunek przodu z chwili wywołania `robot.reset_yaw()`, niezależnie od bieżącej orientacji. Kąt pochodzi z odometrii i jest dostępny z opóźnieniem (telemetria co 50 ms oraz bezwładność silników), dlatego tor jazdy nie będzie idealnie prosty.

Zadanie ujawnia również słabość odometrii niewidoczną przy jeździe po kwadracie: podczas obrotu poślizg kół jest większy niż podczas ruchu postępowego.

**Zadanie 4**

1. Wyznacz `OMEGA_MAX`, czyli prędkość obrotu w rad/s przy `omega = 1.0`, na podstawie dopasowanego `S_MAX` i `L`.
2. Uzupełnij funkcję `rotate(turns, omega=0.3)`, która obraca robota w miejscu o `turns` pełnych obrotów (wartość ujemna oznacza obrót w prawo). Czas obrotu wyznacz z `omega` i `OMEGA_MAX`.
3. Uzupełnij funkcję `drive_turning(heading, distance, velocity, turns)`, która realizuje przejazd na odległość `distance` metrów w kierunku `heading` (w stopniach), w trakcie którego robot obraca się o `turns` obrotów. Czas wynika z odległości i prędkości, a `omega` z czasu i liczby obrotów.

Pierwsza komórka pod zadaniem obraca robota o jeden pełny obrót w miejscu i porównuje kąt pozostały po obrocie według odometrii, LiDAR-u i filtru. Druga wykonuje przejazd 20 cm do przodu z obrotem o pół obrotu, a następnie powrót z obrotem w przeciwną stronę, i wypisuje położenie końcowe względem startu. Przed uruchomieniem ustaw robota na środku pudła.

In [ ]:
OMEGA_MAX = ...


async def rotate(turns, omega=0.3): ...


async def drive_turning(heading, distance, velocity, turns): ...

In [ ]:
view.reset_pose()
await asyncio.sleep(0.5)
before = localizer.estimate
await rotate(1)
await asyncio.sleep(0.5)
after = localizer.estimate
for name in ("odometry", "lidar", "kalman"):
    rest = math.degrees(getattr(after, name).relative_to(getattr(before, name)).yaw)
    print(f"{name:9s} residual after a full turn: {rest:+.1f}°")

In [ ]:
start = localizer.estimate.lidar
robot.field_centric = True
robot.reset_yaw()
await asyncio.sleep(0.3)
try:
    await drive_turning(0, 0.2, 0.15, 0.5)
    await drive_turning(180, 0.2, 0.15, -0.5)
finally:
    robot.field_centric = False
await asyncio.sleep(0.5)
end = localizer.estimate.lidar.relative_to(start)
print(
    f"end relative to start: x = {end.x * 100:+.1f} cm, y = {end.y * 100:+.1f} cm, yaw = {math.degrees(end.yaw):+.1f}°"
)

Podczas przejazdu znaczniki na wykresie obracają się, poruszając się jednocześnie w przybliżeniu po linii prostej. Odchylenie położenia końcowego o kilka centymetrów wynika z opisanego wyżej opóźnienia pomiaru kąta.

## Część 2: Czujniki i filtr Kalmana

### Czujniki ultradźwiękowe

Robot wyposażony jest w cztery rodzaje czujników: enkodery kół, na podstawie których wyznaczana jest odometria, żyroskop [MPU-6050](https://invensense.tdk.com/products/motion-tracking/6-axis/mpu-6050/) mierzący prędkość obrotu, LiDAR oraz czujniki ultradźwiękowe. Enkodery i LiDAR zostały omówione w części 1, a z żyroskopu korzysta filtr położenia opisany na końcu tej części.

Robot ma trzy czujniki ultradźwiękowe [HC-SR04](https://cdn.sparkfun.com/datasheets/Sensors/Proximity/HCSR04.pdf), skierowane do przodu, w lewo i w prawo. Czujnik wysyła impuls o częstotliwości 40 kHz i mierzy czas $t$ do powrotu echa. Dźwięk pokonuje drogę do przeszkody dwukrotnie, zatem odległość wynosi $d = c\,t/2$, gdzie $c \approx 343$ m/s. Zakres pomiaru wynosi od 2 cm do 4 m w stożku o kącie około 15° według noty katalogowej (biblioteka przyjmuje 25°, z zapasem). Czujnik zwraca odległość do najbliższego obiektu w tym stożku.

![HC-SR04: impuls 40 kHz biegnie do ściany i wraca, a pin ECHO pozostaje w stanie wysokim przez czas przelotu](img/ultrasonic_echo.gif)

Czujnik ultradźwiękowy kosztuje kilka złotych, LiDAR kilka tysięcy, co znajduje odzwierciedlenie w jakości pomiaru. Fala odbita od ściany ustawionej pod kątem może nie powrócić do czujnika; wówczas czujnik zwraca maksymalny zasięg albo echo od innego obiektu.

![Pomiar czasu przelotu: echo od ściany prostopadłej i od ściany pod kątem](img/ultrasonic.png)

![Czujnik ultradźwiękowy HC-SR04. Fot. Suyash Dwivedi, CC BY-SA 4.0, Wikimedia Commons](img/hc_sr04_module.jpg)

`robot.us_1.read()` zwraca odczyt przedniego czujnika jako obiekt `Distance` (wartość w metrach w polu `.meters`). Odległość mierzona jest od obudowy czujnika, która znajduje się w odległości `robot.us_1.config.offset` = 15 cm od środka robota.

Sterownik robota nie przesyła obecnie odczytów czujników ultradźwiękowych. W takim przypadku biblioteka emuluje je na podstawie skanu LiDAR-u: wybiera najbliższy punkt w stożku czujnika i dodaje szum oraz sporadyczne fałszywe echa, odwzorowując zachowanie rzeczywistego HC-SR04. Źródło odczytów wskazuje atrybut `robot.us_1.emulated`.

Błędy pomiaru dzielą się na trzy rodzaje, z których każdy wymaga innego postępowania. **Błąd systematyczny (bias)** usuwa się kalibracją, **szum losowy (random noise)** uśrednia filtr, natomiast **pomiary odstające (outliers)** należy wykryć i odrzucić, czym zajmuje się Zadanie 6.

![Trzy rodzaje błędów pomiaru](img/sensor_errors.png)

### Filtr Kalmana w jednym wymiarze

Odometria i czujnik mierzą tę samą wielkość, odległość $d$ środka robota od przedniej ściany, lecz ich błędy mają odmienny charakter. Odometria jest gładka, ale dryfuje; czujnik nie dryfuje, ale każdy jego pomiar jest zaszumiony.

Animacja poniżej przedstawia oba źródła na przykładzie dojazdu do ściany. Odometria (kolor pomarańczowy) oddala się od wartości rzeczywistej, pomiary (kolor szary) rozrzucone są wokół niej, a filtr Kalmana (kolor zielony), implementowany w Zadaniu 5, łączy oba źródła. Pas ±2σ poszerza się w okresie braku pomiarów i zwęża po ich powrocie.

![Filtr Kalmana w 1D: robot jedzie do ściany, odometria dryfuje, czujnik szumi, filtr łączy oba źródła](img/kalman_1d_tracking.gif)

Filtr Kalmana przechowuje **estymatę (estimate)** $\hat d$ oraz jej **wariancję (variance)** $P$, będącą miarą niepewności ($\sqrt P$ to odchylenie standardowe w metrach), i naprzemiennie wykonuje dwa kroki:

* **Predykcja (prediction)**, w każdym kroku. Odometria wskazuje, że robot przemieścił się do przodu o $u$ metrów, zatem ściana jest o $u$ bliżej. Model ruchu nie jest dokładny, więc niepewność rośnie o **szum procesu (process noise)** $Q$:
$$\hat d^- = \hat d - u, \qquad P^- = P + Q$$
* **Korekcja (update)**, po otrzymaniu pomiaru $z$ obarczonego **szumem pomiaru (measurement noise)** o wariancji $R$. **Wzmocnienie Kalmana (Kalman gain)** $K$ określa stopień zaufania do pomiaru, od 0 (pomiar ignorowany) do 1 (pomiar przyjmowany bez zmian):
$$K = \frac{P^-}{P^- + R}, \qquad \hat d = \hat d^- + K\,(z - \hat d^-), \qquad P = (1 - K)\,P^-$$

![Pętla filtru Kalmana](img/kalman_loop.png)

Oba kroki można przedstawić na rozkładach prawdopodobieństwa. Predykcja przesuwa rozkład estymaty i go poszerza, a korekcja mnoży go przez rozkład pomiaru. Wynikowy rozkład leży pomiędzy nimi, bliżej źródła o mniejszej wariancji, i jest węższy od obu.

![Predykcja i korekcja na rozkładach normalnych dla dokładnego i dla silnie zaszumionego pomiaru](img/kalman_gauss.gif)

Jest to najprostszy, jednowymiarowy przypadek filtru opisanego przez Kalmana w 1960 roku. Przystępne wprowadzenie zawiera artykuł [How a Kalman filter works, in pictures](https://www.bzarg.com/p/how-a-kalman-filter-works-in-pictures/), a szczegółowe wyprowadzenie strona [kalmanfilter.net](https://www.kalmanfilter.net/kalman1d.html):

> Kalman, Rudolph Emil. *"A new approach to linear filtering and prediction problems."* Journal of Basic Engineering 82.1 (1960): 35-45. [link](https://www.unitedthc.com/DSP/Kalman1960.pdf)

Pierwszym krokiem jest nagranie dojazdu do ściany. Komórka cofa robota o 20 cm, następnie powoli przejeżdża 40 cm do przodu i wraca na środek pudła. Podczas jazdy do przodu co 0,1 s zapisywane są: odczyt przedniego czujnika (przeliczony na odległość środka robota od ściany), położenie $x$ z odometrii oraz odległość od ściany zmierzona LiDAR-em, przyjmowana jako wartość referencyjna. `LivePlot` rysuje na żywo dowolne wielkości przekazane jako funkcje i zapisuje je w `rec.data`. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
US_OFFSET = robot.us_1.config.offset


def lidar_front():
    pts = robot.lidar_scan.points_in_meters()
    ahead = pts[(np.abs(pts[:, 1]) < 0.02) & (pts[:, 0] > 0)]
    return float(np.median(ahead[:, 0])) if len(ahead) else None


await robot.drive_for(DriveVector(0.3, Angle.deg(180)), Time.s(0.2 / (0.3 * S_MAX)))
view.reset_pose()
await asyncio.sleep(0.5)

rec = LivePlot("Approach to the wall", y_label="distance to the wall [m]", y_range=(0, 0.8))
rec.add("ultrasonic", lambda: robot.us_1.read().meters + US_OFFSET, kind="dots")
rec.add("odometry x", lambda: localizer.estimate.odometry.x)
rec.add("LiDAR", lidar_front)
rec.show()
await robot.drive_for(DriveVector(0.15, Angle.deg(0)), Time.s(0.4 / (0.15 * S_MAX)))
await asyncio.sleep(0.5)
rec.stop()
await robot.drive_for(DriveVector(0.3, Angle.deg(180)), Time.s(0.2 / (0.3 * S_MAX)))

data = rec.data.dropna(subset=["odometry x", "LiDAR"])
t, z, truth = data["t"].to_numpy(), data["ultrasonic"].to_numpy(), data["LiDAR"].to_numpy()
u = np.diff(data["odometry x"].to_numpy(), prepend=data["odometry x"].iloc[0])
print(f"samples: {len(data)}, distance from {truth[0]:.2f} m to {truth[-1]:.2f} m")

**Zadanie 5**

Uzupełnij funkcję `kalman_1d(u, z, d0, P0, Q, R)`, która przetwarza nagranie filtrem Kalmana. W każdym kroku $k$ wykonaj predykcję z przyrostem odometrii `u[k]`, a jeśli `z[k]` nie jest wartością `nan` (`np.isnan`), również korekcję. Funkcja zwraca trzy tablice: estymaty $\hat d$, ich odchylenia standardowe $\sqrt P$ po każdym kroku oraz informację, czy w danym kroku odrzucono pomiar. Argument `n_sigma` zostanie wykorzystany w Zadaniu 6; na tym etapie żaden pomiar nie jest odrzucany.

In [ ]:
def kalman_1d(u, z, d0, P0, Q, R, n_sigma=None):
    d, P = d0, P0
    estimates, stds, rejected = [], [], []
    ...
    return np.array(estimates), np.array(stds), np.array(rejected)

Komórka poniżej uruchamia filtr z szumem pomiaru $R = (1\text{ cm})^2$ i szumem procesu $Q = (2\text{ mm})^2$ na krok, rysuje wynik i wyznacza **pierwiastek błędu średniokwadratowego (root mean squared error, RMSE)** każdego źródła względem LiDAR-u. Wynik porównywany jest również z implementacją z biblioteki pykalman; przy poprawnej implementacji obie krzywe się pokrywają. Następnie zmień wartości $Q$ i $R$ o rząd wielkości w obu kierunkach i zaobserwuj wpływ na estymatę.

In [ ]:
from omni_drive.ui import rmse
from pykalman import KalmanFilter

Q, R = 0.002**2, 0.01**2
d0 = float(np.nanmedian(z[:5]))
est, std, _ = kalman_1d(u, z, d0, R, Q, R)

reference = KalmanFilter(
    transition_matrices=[[1.0]],
    observation_matrices=[[1.0]],
    transition_offsets=-u[1:, None],
    transition_covariance=[[Q]],
    observation_covariance=[[R]],
    initial_state_mean=[d0 - u[0]],
    initial_state_covariance=[[R + Q]],
)
ref_est = reference.filter(np.ma.masked_invalid(z[:, None]))[0][:, 0]

odometry_only = d0 - np.cumsum(u)
plot_series(
    t,
    {
        "sensor": z,
        "odometry": odometry_only,
        "LiDAR (reference)": truth,
        "Kalman filter": est,
        "pykalman": ref_est,
    },
    dots=["sensor"],
    band=(est - 2 * std, est + 2 * std, "±2σ"),
    title="1D Kalman filter",
    y_label="distance to the wall [m]",
)
for name, values in {"odometry": odometry_only, "sensor": z, "Kalman filter": est}.items():
    print(f"RMSE {name}: {rmse(values, truth) * 100:.1f} cm")

### Odrzucanie pomiarów odstających

Filtr Kalmana zakłada, że szum pomiaru ma rozkład normalny. Fałszywe echo, np. odczyt 4 m przy ścianie oddalonej o 30 cm, odbiega od wartości rzeczywistej o kilkaset odchyleń standardowych. Rozkład normalny przypisuje takiemu odchyleniu prawdopodobieństwo bliskie zeru, mimo to filtr przyjmuje pomiar, a estymata gwałtownie się zmienia.

Zapobiega temu **bramkowanie innowacji (innovation gating)**, czyli odrzucanie pomiarów przed korekcją. **Innowacja (innovation)** $y = z - \hat d^-$ ma wariancję $S = P^- + R$, będącą sumą niepewności estymaty i pomiaru. Pomiar zostaje odrzucony, jeżeli jego odległość od predykcji przekracza $n_\sigma$ odchyleń standardowych:

$$
y^2 > n_\sigma^2 \cdot S
$$

Typowo przyjmuje się $n_\sigma = 3$. Wielkość $y^2 / S$ jest kwadratem [odległości Mahalanobisa](https://pl.wikipedia.org/wiki/Odleg%C5%82o%C5%9B%C4%87_Mahalanobisa) pomiaru od predykcji. W przypadku wielowymiarowym porównuje się ją z kwantylem rozkładu $\chi^2$; w ten sposób działa filtr w bibliotece.

![Bramkowanie innowacji: pomiary mieszczące się w bramce poprawiają estymatę, fałszywe echo zostaje odrzucone](img/gating.gif)

**Zadanie 6**

1. Uzupełnij funkcję `kalman_1d` z Zadania 5 o bramkowanie: jeżeli `n_sigma` nie jest `None`, a pomiar nie mieści się w bramce, pomiń korekcję i zapisz informację o odrzuceniu pomiaru. Uruchom ponownie komórkę z funkcją.
2. Uruchom filtr z `n_sigma=3`, wyznacz liczbę odrzuconych pomiarów i porównaj RMSE z wynikiem bez bramkowania.

In [ ]:
...

In [ ]:
plot_series(
    t,
    {
        "sensor": z,
        "LiDAR (reference)": truth,
        "Kalman filter": est,
        "Kalman filter with gating": gated,
    },
    dots=["sensor"],
    title="3σ gating",
    y_label="distance to the wall [m]",
)

### Filtr Kalmana dla położenia robota

Filtr zaimplementowany w bibliotece działa na tej samej zasadzie, przy czym jego stanem jest pełne położenie $\mathbf{x} = [x, y, \theta]^T$, a niepewność opisuje macierz kowariancji $\mathbf{P}$ (3 × 3). Predykcja wykorzystuje ten sam wzór co funkcja `odometry_step` z Zadania 1 (obrót przyrostu o kąt w połowie kroku) i wykonywana jest dla każdego przyrostu odometrii, czyli co 50 ms. Kąt obrotu pochodzi z żyroskopu, ponieważ, jak pokazało Zadanie 4, odometria kół jest najmniej dokładna właśnie podczas obrotów. Korekcja wykorzystuje położenie z LiDAR-u jako pomiar $\mathbf{z}$ i stosuje bramkowanie analogiczne do Zadania 6, w trzech wymiarach (próg $\chi^2 = 14{,}16$).

Ruch zależy od kąta $\theta$ nieliniowo, poprzez funkcje $\sin$ i $\cos$. Biblioteka stosuje więc **rozszerzony filtr Kalmana (extended Kalman filter, EKF)**, który w każdym kroku linearyzuje model ruchu macierzą pochodnych $\mathbf{F}$ i propaguje niepewność zgodnie ze wzorem $\mathbf{P}^- = \mathbf{F}\mathbf{P}\mathbf{F}^T + \mathbf{Q}$. Pełny opis zawierają rozdziały 3 i 7 książki [Probabilistic Robotics](http://robots.stanford.edu/probabilistic-robotics/) (S. Thrun, W. Burgard, D. Fox) oraz [Kalman and Bayesian Filters in Python](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python) (R. Labbe). Implementacja, licząca około stu linii, znajduje się w pliku `omni_drive/kalman.py`.

![Trzy estymaty położenia na kwadratowej trasie: odometria dryfuje, LiDAR jest zaszumiony, a elipsa filtru rośnie między pomiarami i maleje po otrzymaniu położenia z LiDAR-u](img/pose_estimates.gif)

Filtr konfiguruje się obiektem `PoseNoise`, którego pola są odchyleniami standardowymi. Szum procesu, odzwierciedlający zaufanie do modelu ruchu, opisują pola `per_meter` (błąd położenia z odometrii po przejechaniu 1 m) oraz `yaw_per_meter` i `yaw_per_rad` (błąd kąta po przejechaniu 1 m i po obrocie o 1 rad). Szum pomiaru, odzwierciedlający zaufanie do LiDAR-u, opisują pola `lidar_xy` (położenie) i `lidar_yaw` (kąt).

Wartości domyślne są przybliżone. Poprawnie skonfigurowany filtr wyznacza niepewność zgodną z rzeczywistym błędem: prawdziwe położenie powinno znajdować się wewnątrz elipsy 2σ przez około 86% czasu. W dwóch wymiarach elipsa 2σ obejmuje 86% rozkładu, a nie 95% jak przedział ±2σ w jednym wymiarze.

**Zadanie 7**

1. Komórka przez 5 s rejestruje estymaty nieruchomego robota (nie należy go w tym czasie dotykać). Na podstawie nagrania (`localizer.dataframe(since=...)`) wyznacz odchylenie standardowe położenia z LiDAR-u: `lidar_xy` jako średnią odchyleń kolumn `lidar_x` i `lidar_y` oraz `lidar_yaw` z kolumny `lidar_yaw`.
2. Na podstawie wyniku Zadania 3 oszacuj `per_meter` jako błąd odometrii podzielony przez pierwiastek z przejechanej drogi (3 okrążenia kwadratu o boku 20 cm to 2,4 m; wariancja błędu rośnie liniowo z drogą). Podczas postoju LiDAR wykazuje mniejszy błąd niż w ruchu, dlatego zwiększ `lidar_xy` i `lidar_yaw` 3-5 razy, nie przyjmując wartości mniejszych niż 5 mm i 0,5°.
3. Utwórz obiekt `PoseNoise` z wyznaczonymi wartościami i nowy `Localizer(robot, noise=...)`. Komórka pod zadaniem wykona przejazd po kwadracie i wyświetli trzy estymaty położenia oraz wykresy czujników ultradźwiękowych.

In [ ]:
start = localizer.estimate.t
await asyncio.sleep(5.0)
still = localizer.dataframe(since=start)

...

`UltrasonicView` przedstawia dla każdego czujnika jego odczyty (szare punkty) oraz wartości, jakie czujnik powinien zmierzyć, gdyby robot znajdował się w położeniu wskazywanym przez odometrię, LiDAR albo filtr. Odczyt oczekiwany jest odległością do najbliższej zajętej komórki mapy w stożku czujnika.

Czujniki ultradźwiękowe nie biorą udziału w wyznaczaniu położenia, dlatego stanowią niezależną weryfikację: im bliżej linii leżą punkty, tym dokładniejsza estymata. Jeżeli jednak `robot.us_1.emulated` ma wartość `True`, odczyty pochodzą z tego samego skanu co LiDAR i z definicji będą zgodne z estymatą LiDAR-u; w pełni niezależną weryfikację zapewniają dopiero fizyczne czujniki HC-SR04. Pod wykresami wyświetlana jest mediana odległości punktów od każdej linii. Mediana, w odróżnieniu od RMSE, jest odporna na pojedyncze fałszywe echa. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
await localizer.wait_ready()
view = LidarView(localizer=localizer).show()
sonar = UltrasonicView(localizer=localizer).show()
for _ in range(3):
    await drive_square(0.2, 0.3)

In [ ]:
ok = sonar.data[sonar.data["measured"] < robot.us_1.config.max_range]
gaps = ok[["odometry", "lidar", "kalman"]].sub(ok["measured"], axis=0).abs()
print("median |measured - expected| [cm]:")
print((gaps.groupby(ok["sensor"]).median() * 100).round(1))

Ostatnia komórka wykorzystuje czujnik ultradźwiękowy w jego typowym zastosowaniu. Robot jedzie do przodu, a metoda `wait_for` oczekuje, aż przedni czujnik zmierzy odległość mniejszą niż 10 cm. Limit czasu `safe_seconds` odpowiada 30 cm jazdy, dzięki czemu robot nie dojedzie do ściany nawet w przypadku awarii czujnika. Blok `finally` zatrzymuje robota również po przerwaniu komórki. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
robot.set_vector(DriveVector(0.2, Angle.deg(0)))
try:
    safe_seconds = 0.3 / (0.2 * S_MAX)
    hit = await asyncio.wait_for(robot.us_1.wait_for(lambda d: d.meters < 0.10), safe_seconds)
    print(f"stopped at a reading of {hit.meters * 100:.1f} cm")
finally:
    robot.stop()

## Materiały dodatkowe

* [Kalman and Bayesian Filters in Python](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python) - bezpłatna książka w formie notebooków, od filtru g-h przez EKF do filtru cząsteczkowego.
* [Probabilistic Robotics](http://robots.stanford.edu/probabilistic-robotics/) - podręcznik lokalizacji i SLAM.
* [robot_localization](https://docs.ros.org/en/melodic/api/robot_localization/html/index.html) - implementacja EKF w ROS, łącząca odometrię, IMU i inne czujniki na tej samej zasadzie co `Localizer`.
* [Dokumentacja `omni_drive`](https://omni-drive.readthedocs.io/) z opisem wszystkich klas wykorzystanych w notebooku. Robot nie ma dostępu do internetu, dlatego dokumentację należy otworzyć na własnym komputerze.